# RFM and Cohort Analysis: Customer Segmentation and Retention

**Goal:** Segment customers using RFM (Recency, Frequency, Monetary) and analyse retention trends using Cohort Analysis.

**Dataset:** UCI Online Retail (auto-downloaded). Optional: Online Retail II.

**Notebook steps**
1. Setup and imports
2. Download / load dataset
3. Data cleaning and preprocessing
4. Exploratory Data Analysis (EDA)
5. RFM metric calculation
6. RFM scoring and customer segments
7. RFM visualizations
8. Cohort Analysis (retention)
9. Cohort visualizations
10. Business insights
11. Export outputs

## Step 1: Setup and imports

In [ ]:
# Run once if libraries are missing (remove the # to run)
# !pip install pandas numpy matplotlib seaborn openpyxl

import os, io, glob, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.options.display.float_format = "{:,.2f}".format

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("Setup done")

## Step 2: Download / load the dataset

The cell below tries these sources in order:
1. A file you already placed in `data/` (`data.csv` or any `.xlsx`)
2. Auto-download from UCI Machine Learning Repository
3. Kaggle mirror through `kagglehub` (optional)
4. A **synthetic** dataset (only so the notebook still runs; use a real dataset for your submission)

Change `DATASET` to `"online_retail_ii"` for the bigger 2-year dataset.

In [ ]:
DATASET = "online_retail"      # or "online_retail_ii"

UCI = {
    "online_retail": ("https://archive.ics.uci.edu/static/public/352/online+retail.zip", "Online Retail.xlsx"),
    "online_retail_ii": ("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", "online_retail_II.xlsx"),
}

def load_local():
    csvs = glob.glob("data/*.csv")
    xlsx = glob.glob("data/*.xlsx")
    if csvs:
        print("Loading local CSV:", csvs[0])
        return pd.read_csv(csvs[0], encoding="ISO-8859-1")
    if xlsx:
        print("Loading local Excel:", xlsx[0])
        sheets = pd.read_excel(xlsx[0], sheet_name=None)
        return pd.concat(sheets.values(), ignore_index=True)
    return None

def load_uci(key):
    url, fname = UCI[key]
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    print("Downloading from UCI ...")
    data = urllib.request.urlopen(req, timeout=180).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall("data")
    return load_local()

def load_kaggle():
    import kagglehub   # pip install kagglehub
    folder = kagglehub.dataset_download("carrie1/ecommerce-data")
    f = glob.glob(os.path.join(folder, "*.csv"))[0]
    return pd.read_csv(f, encoding="ISO-8859-1")

def make_synthetic(n_customers=2500, seed=42):
    # Same columns as UCI Online Retail, so every later step works unchanged
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2010-12-01")
    rows, inv = [], 500000
    for cid in range(12000, 12000 + n_customers):
        first = start + pd.Timedelta(days=int(rng.integers(0, 330)))
        n_orders = int(rng.geometric(0.28))
        d = first
        for _ in range(n_orders):
            if d > pd.Timestamp("2011-12-09"):
                break
            inv += 1
            for _ in range(int(rng.integers(1, 6))):
                rows.append([str(inv), f"P{rng.integers(1000, 1300)}", "SYNTHETIC ITEM",
                             int(rng.integers(1, 12)), d + pd.Timedelta(hours=int(rng.integers(8, 18))),
                             round(float(rng.uniform(0.5, 25)), 2), cid,
                             rng.choice(["United Kingdom", "Germany", "France", "EIRE", "Spain"], p=[.82, .07, .05, .03, .03])])
            d += pd.Timedelta(days=int(rng.integers(5, 90)))
    df = pd.DataFrame(rows, columns=["InvoiceNo", "StockCode", "Description", "Quantity",
                                     "InvoiceDate", "UnitPrice", "CustomerID", "Country"])
    # add dirty rows so the cleaning step has something to do
    junk = df.sample(200, random_state=1).copy(); junk["CustomerID"] = np.nan
    canc = df.sample(150, random_state=2).copy(); canc["InvoiceNo"] = "C" + canc["InvoiceNo"]; canc["Quantity"] *= -1
    return pd.concat([df, junk, canc, df.head(50)], ignore_index=True)

df_raw, source = load_local(), "local file"
if df_raw is None:
    try:
        df_raw, source = load_uci(DATASET), "UCI"
    except Exception as e:
        print("UCI download failed:", e)
if df_raw is None:
    try:
        df_raw, source = load_kaggle(), "Kaggle (kagglehub)"
    except Exception as e:
        print("Kaggle download failed:", e)
if df_raw is None:
    print("WARNING: using SYNTHETIC data. Download the real dataset and put it in data/ for your submission.")
    df_raw, source = make_synthetic(), "SYNTHETIC"

# Standardise column names (Online Retail II uses different names)
df_raw = df_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
print("Data source:", source)
print("Shape:", df_raw.shape)
df_raw.head()

## Step 3: Data cleaning and preprocessing

Problems handled (write these in your README):
- Missing `CustomerID` (cannot assign purchases to a customer)
- Duplicate rows
- Cancelled orders (invoice starts with `C`)
- Zero or negative quantity / price
- Wrong data types
- Creating `TotalPrice = Quantity x UnitPrice`

In [ ]:
print("Missing values before cleaning:")
print(df_raw.isna().sum(), "\n")

df = df_raw.copy()
log = {"raw rows": len(df)}

df = df.dropna(subset=["CustomerID"]);                      log["after dropping missing CustomerID"] = len(df)
df = df.drop_duplicates();                                  log["after dropping duplicates"] = len(df)
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")];  log["after removing cancelled orders"] = len(df)
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)];      log["after removing qty/price <= 0"] = len(df)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

pd.Series(log, name="rows").to_frame()

In [ ]:
print("Date range :", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Customers  :", df["CustomerID"].nunique())
print("Invoices   :", df["InvoiceNo"].nunique())
print("Countries  :", df["Country"].nunique())
print("Total revenue:", round(df["TotalPrice"].sum(), 2))
df.describe()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Monthly revenue trend
monthly = df.set_index("InvoiceDate").resample("MS")["TotalPrice"].sum()
ax = monthly.plot(marker="o", color="#2a6fdb")
ax.set_title("Monthly Revenue Trend"); ax.set_ylabel("Revenue"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig("outputs/01_monthly_revenue.png", dpi=150); plt.show()

In [ ]:
# 4.2 Top 10 countries by revenue
top_c = df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_c.values, y=top_c.index, color="#2a6fdb")
plt.title("Top 10 Countries by Revenue"); plt.xlabel("Revenue"); plt.ylabel("")
plt.tight_layout(); plt.savefig("outputs/02_top_countries.png", dpi=150); plt.show()

In [ ]:
# 4.3 Top 10 products by quantity sold
top_p = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_p.values, y=top_p.index, color="#e07b39")
plt.title("Top 10 Products by Quantity"); plt.xlabel("Units sold"); plt.ylabel("")
plt.tight_layout(); plt.savefig("outputs/03_top_products.png", dpi=150); plt.show()

In [ ]:
# 4.4 Purchase behaviour: orders per customer, and weekday / hour pattern
orders_per_cust = df.groupby("CustomerID")["InvoiceNo"].nunique()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
orders_per_cust.clip(upper=30).plot(kind="hist", bins=30, ax=ax[0], color="#2a6fdb")
ax[0].set_title("Orders per Customer (capped at 30)")

df["Weekday"] = df["InvoiceDate"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df.groupby("Weekday")["InvoiceNo"].nunique().reindex(order).plot(kind="bar", ax=ax[1], color="#2a6fdb")
ax[1].set_title("Orders by Weekday"); ax[1].set_xlabel("")

df.groupby(df["InvoiceDate"].dt.hour)["InvoiceNo"].nunique().plot(kind="bar", ax=ax[2], color="#2a6fdb")
ax[2].set_title("Orders by Hour"); ax[2].set_xlabel("Hour")
plt.tight_layout(); plt.savefig("outputs/04_purchase_patterns.png", dpi=150); plt.show()

print("Repeat customers (2+ orders): {:.1%}".format((orders_per_cust > 1).mean()))

## Step 5: RFM metric calculation

- **Recency (R):** days since the customer's last purchase (lower is better)
- **Frequency (F):** number of unique orders
- **Monetary (M):** total money spent

The snapshot date is one day after the last transaction in the data.

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date.date())

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
).reset_index()

rfm.describe()

## Step 6: RFM scoring and segments

Each metric gets a score of 1 to 5 using quintiles (5 is best). For Recency the labels are reversed because a smaller number of days is better. `rank(method="first")` avoids errors when many customers share the same value.

In [ ]:
rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

def assign_segment(row):
    r, f = row["R_Score"], row["F_Score"]
    if r >= 4 and f >= 4: return "Champions"
    if r >= 3 and f >= 4: return "Loyal Customers"
    if r >= 4 and f <= 2: return "New Customers"
    if r >= 3 and f == 3: return "Potential Loyalists"
    if r == 3 and f <= 2: return "Promising"
    if r <= 2 and f >= 4: return "Cannot Lose Them"
    if r <= 2 and f == 3: return "At Risk"
    if r == 2 and f <= 2: return "About to Sleep"
    return "Lost"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)
rfm.head(10)

In [ ]:
seg = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
).sort_values("Total_Revenue", ascending=False)
seg["Customer_%"] = seg["Customers"] / seg["Customers"].sum() * 100
seg["Revenue_%"] = seg["Total_Revenue"] / seg["Total_Revenue"].sum() * 100
seg.round(1)

## Step 7: RFM visualizations

In [ ]:
palette = sns.color_palette("viridis", len(seg))
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
seg["Customers"].sort_values().plot(kind="barh", ax=ax[0], color=palette)
ax[0].set_title("Customers per Segment"); ax[0].set_ylabel("")
seg["Revenue_%"].sort_values().plot(kind="barh", ax=ax[1], color=palette)
ax[1].set_title("Revenue Share per Segment (%)"); ax[1].set_ylabel("")
plt.tight_layout(); plt.savefig("outputs/05_rfm_segments.png", dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x="Recency", y="Monetary", hue="Segment", alpha=0.7, s=35)
plt.yscale("log"); plt.title("Recency vs Monetary by Segment")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("outputs/06_recency_vs_monetary.png", dpi=150); plt.show()

In [ ]:
# Heatmap: average Monetary value by R and F score
pivot = rfm.pivot_table(index="R_Score", columns="F_Score", values="Monetary", aggfunc="mean")
sns.heatmap(pivot.sort_index(ascending=False), annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Average Monetary Value by R and F Score")
plt.tight_layout(); plt.savefig("outputs/07_rf_heatmap.png", dpi=150); plt.show()

## Step 8: Cohort Analysis

- **Cohort month:** month of a customer's first purchase
- **Cohort index:** how many months after the first purchase an order happened (1 = first month)
- **Retention rate:** customers active in month N / customers in the cohort

In [ ]:
df["OrderMonth"] = df["InvoiceDate"].dt.to_period("M")
df["CohortMonth"] = df.groupby("CustomerID")["InvoiceDate"].transform("min").dt.to_period("M")

df["CohortIndex"] = ((df["OrderMonth"].dt.year - df["CohortMonth"].dt.year) * 12
                     + (df["OrderMonth"].dt.month - df["CohortMonth"].dt.month) + 1)

cohort_counts = (df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"]
                   .nunique().reset_index()
                   .pivot(index="CohortMonth", columns="CohortIndex", values="CustomerID"))

cohort_size = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_size, axis=0)

print("Cohort sizes (customers acquired per month):")
cohort_size.to_frame("Customers")

In [ ]:
cohort_counts

In [ ]:
(retention * 100).round(1)

## Step 9: Cohort visualizations

In [ ]:
plt.figure(figsize=(13, 7))
sns.heatmap(retention, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=0.6,
            cbar_kws={"label": "Retention rate"})
plt.title("Monthly Cohort Retention"); plt.xlabel("Months since first purchase"); plt.ylabel("Cohort month")
plt.tight_layout(); plt.savefig("outputs/08_cohort_retention_heatmap.png", dpi=150); plt.show()

In [ ]:
# Retention curves: one line per cohort + the average
plt.figure(figsize=(11, 6))
for cohort_month, row in retention.iterrows():
    plt.plot(row.index, row.values, alpha=0.35, color="grey")
avg_ret = retention.mean()
plt.plot(avg_ret.index, avg_ret.values, color="#d62728", linewidth=3, marker="o", label="Average")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.title("Cohort Retention Curves"); plt.xlabel("Months since first purchase"); plt.ylabel("Retention rate")
plt.legend(); plt.tight_layout(); plt.savefig("outputs/09_retention_curves.png", dpi=150); plt.show()

In [ ]:
# Average spend per active customer by cohort and month (extra insight)
spend = (df.groupby(["CohortMonth", "CohortIndex"])["TotalPrice"].sum()
           .div(df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"].nunique())
           .reset_index(name="AvgSpend")
           .pivot(index="CohortMonth", columns="CohortIndex", values="AvgSpend"))
plt.figure(figsize=(13, 7))
sns.heatmap(spend, annot=True, fmt=".0f", cmap="Greens")
plt.title("Average Spend per Active Customer by Cohort")
plt.tight_layout(); plt.savefig("outputs/10_cohort_avg_spend.png", dpi=150); plt.show()

## Step 10: Business insights

In [ ]:
champ = seg.loc["Champions"] if "Champions" in seg.index else None
lost = seg.loc["Lost"] if "Lost" in seg.index else None
risk = seg.loc[[s for s in ["At Risk", "Cannot Lose Them"] if s in seg.index]]

print("KEY INSIGHTS (numbers come from your data)")
print("-" * 60)
if champ is not None:
    print(f"1. Champions are {champ['Customer_%']:.1f}% of customers but bring {champ['Revenue_%']:.1f}% of revenue.")
top2 = seg.sort_values("Revenue_%", ascending=False).head(2)
print(f"2. Top two revenue segments: {', '.join(top2.index)} ({top2['Revenue_%'].sum():.1f}% of revenue).")
print(f"3. Average retention in month 2: {avg_ret.get(2, float('nan')):.1%}; month 3: {avg_ret.get(3, float('nan')):.1%}; "
      f"month 6: {avg_ret.get(6, float('nan')):.1%}.")
drop = (1 - avg_ret.get(2, np.nan))
print(f"4. {drop:.1%} of new customers do not come back in the month after their first purchase.")
print(f"5. {(orders_per_cust > 1).mean():.1%} of customers purchased more than once.")
if len(risk):
    print(f"6. At Risk / Cannot Lose Them: {int(risk['Customers'].sum())} customers worth "
          f"{risk['Revenue_%'].sum():.1f}% of revenue; target them with win-back offers.")
if lost is not None:
    print(f"7. Lost segment: {lost['Customer_%']:.1f}% of customers, only {lost['Revenue_%']:.1f}% of revenue; low priority.")

### Recommendations (adjust numbers using your own output)

| Segment / finding | Action |
|---|---|
| **Champions** | Loyalty programme, early access to new products, referral rewards |
| **Loyal Customers** | Upsell and cross-sell, bundle offers |
| **New Customers / Promising** | Welcome series, second-purchase discount within 30 days |
| **At Risk / Cannot Lose Them** | Personalised win-back emails, limited-time offers |
| **Lost** | Low-cost reactivation campaign only |
| **Big drop after month 1** | Improve onboarding, follow-up emails and reminders in the first 30 days |

## Step 11: Export outputs

In [ ]:
rfm.to_csv("outputs/rfm_customers.csv", index=False)
seg.round(2).to_csv("outputs/rfm_segment_summary.csv")
retention.round(4).to_csv("outputs/cohort_retention_table.csv")
cohort_counts.to_csv("outputs/cohort_counts_table.csv")
print("Saved files:")
for f in sorted(os.listdir("outputs")):
    print(" -", f)